# Domain Adaptation & Unpaired Image-to-Image Translation using CycleGAN

**Objective:** Design and implement an unpaired image-to-image translation system using CycleGAN to learn mappings between two domains (Sketch ↔ Photo) without paired data.

**Key Features:**
- Translate Sketch → Photo (G_AB)
- Translate Photo → Sketch (G_BA)
- Preserve structural consistency using cycle constraints

**Architecture:**
- Generator: ResNet-based (6 blocks), 128×128 images
- Discriminator: PatchGAN
- Training: Mixed Precision (AMP), Adam optimizer, LSGAN loss

**Datasets:**
- TU-Berlin Sketch Dataset (HuggingFace) — Domain A
- Sketchy Dataset (Kaggle) — Domain A
- Google QuickDraw Dataset (Kaggle) — Domain A
- STL-10 (torchvision) — Domain B (Photo)

---
## 1. Environment Setup

In [ ]:
!pip install -q datasets gradio scikit-image

In [ ]:
import os
import glob
import random
import itertools
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import STL10

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'Device: {device} | GPUs available: {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} '
          f'({torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB)')

In [ ]:
# ========================
# Configuration
# ========================
CONFIG = {
    'img_size': 128,
    'batch_size': 8,         # Doubled for dual-GPU
    'num_epochs': 30,
    'lr': 0.0002,
    'betas': (0.5, 0.999),
    'lambda_cycle': 10.0,
    'lambda_identity': 5.0,
    'n_resnet_blocks': 6,
    'ngf': 64,
    'ndf': 64,
    'in_channels': 3,
    'out_channels': 3,
    'decay_epoch': 15,       # Start LR decay at epoch 15
    'sample_interval': 5,
    'checkpoint_interval': 10,
    'num_workers': 2,
    'buffer_size': 50,
    'max_sketches': 3000,    # Max sketches per-source
    'max_photos': 4000,      # Max photos from STL-10
}

# Kaggle paths
SKETCHY_PATH = '/kaggle/input/sketchy-dataset'
TUBERLIN_HF_ID = 'sdiaeyu6n/tu-berlin'
QUICKDRAW_PATH = '/kaggle/input/quickdraw-doodle-recognition'

OUTPUT_DIR = '/kaggle/working/cyclegan_output'
CHECKPOINT_DIR = f'{OUTPUT_DIR}/checkpoints'
SAMPLE_DIR = f'{OUTPUT_DIR}/samples'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

---
## 2. Data Preparation

**Domain A (Sketches):** TU-Berlin + Sketchy + QuickDraw  
**Domain B (Photos):** STL-10 (real-world object photos)

> All three assigned datasets are sketch/doodle-only. For the photo domain, we use STL-10 which provides real-world photos with overlapping categories (airplane, bird, car, cat, dog, horse, etc.).

In [ ]:
# ========================
# Utility: find images recursively
# ========================
def find_images(directory, extensions=None):
    if extensions is None:
        extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif'}
    images = []
    for root, dirs, files in os.walk(directory):
        for f in sorted(files):
            if os.path.splitext(f)[1].lower() in extensions:
                images.append(os.path.join(root, f))
    return images


def render_quickdraw(drawing_str, size=256):
    """Render QuickDraw stroke data to a PIL image."""
    import ast
    strokes = ast.literal_eval(drawing_str)
    img = Image.new('RGB', (size, size), 'white')
    draw = ImageDraw.Draw(img)
    for stroke in strokes:
        if len(stroke) >= 2:
            points = list(zip(stroke[0], stroke[1]))
            if len(points) > 1:
                draw.line(points, fill='black', width=3)
    return img


print('Utilities defined.')

In [ ]:
# ========================
# Collect Domain A: Sketches from ALL 3 datasets
# ========================
domain_A_paths = []

# --- 1) TU-Berlin Sketch Dataset (HuggingFace) ---
tu_berlin_dir = '/kaggle/working/tu_berlin_images'
os.makedirs(tu_berlin_dir, exist_ok=True)

try:
    from datasets import load_dataset
    n_tu = min(CONFIG['max_sketches'], 3000)
    tu_berlin = load_dataset(TUBERLIN_HF_ID, split=f'train[:{n_tu}]')
    for i, sample in enumerate(tu_berlin):
        img = sample['image'].convert('RGB')
        save_path = f'{tu_berlin_dir}/{i:05d}.png'
        img.save(save_path)
        domain_A_paths.append(save_path)
    print(f'[Domain A] TU-Berlin: {len(domain_A_paths)} sketches loaded')
except Exception as e:
    print(f'[Domain A] TU-Berlin failed: {e}')

# --- 2) Sketchy Dataset (Kaggle) ---
if os.path.exists(SKETCHY_PATH):
    sketchy_imgs = find_images(SKETCHY_PATH)
    if sketchy_imgs:
        domain_A_paths.extend(sketchy_imgs[:CONFIG['max_sketches']])
        print(f'[Domain A] Sketchy: {min(len(sketchy_imgs), CONFIG["max_sketches"])} images added')
    else:
        print('[Domain A] Sketchy: no image files found (dataset is code-only)')
else:
    print(f'[Domain A] Sketchy: not found at {SKETCHY_PATH}')

# --- 3) Google QuickDraw Dataset (Kaggle) ---
if os.path.exists(QUICKDRAW_PATH):
    import pandas as pd
    import ast
    
    quickdraw_dir = '/kaggle/working/quickdraw_images'
    os.makedirs(quickdraw_dir, exist_ok=True)
    
    csv_dirs = [
        os.path.join(QUICKDRAW_PATH, 'train_simplified'),
        os.path.join(QUICKDRAW_PATH, 'train_raw'),
        QUICKDRAW_PATH,
    ]
    csv_files = []
    for d in csv_dirs:
        csv_files = sorted(glob.glob(os.path.join(d, '*.csv')))
        if csv_files:
            break
    
    qd_count = 0
    max_per_file = 100
    max_total = CONFIG['max_sketches']
    
    for csv_file in csv_files:
        if qd_count >= max_total:
            break
        try:
            df = pd.read_csv(csv_file, nrows=max_per_file)
            drawing_col = None
            for col in df.columns:
                if 'drawing' in col.lower():
                    drawing_col = col
                    break
            if drawing_col is None:
                continue
            for _, row in df.iterrows():
                if qd_count >= max_total:
                    break
                try:
                    img = render_quickdraw(row[drawing_col])
                    save_path = f'{quickdraw_dir}/{qd_count:05d}.png'
                    img.save(save_path)
                    domain_A_paths.append(save_path)
                    qd_count += 1
                except Exception:
                    continue
        except Exception:
            continue
    print(f'[Domain A] QuickDraw: {qd_count} doodle images rendered')
else:
    print(f'[Domain A] QuickDraw: not found at {QUICKDRAW_PATH}')

print(f'\n>>> Total Domain A (Sketches): {len(domain_A_paths)}')

In [ ]:
# ========================
# Collect Domain B: Photos from STL-10
# ========================
domain_B_paths = []
stl10_dir = '/kaggle/working/stl10_photos'
os.makedirs(stl10_dir, exist_ok=True)

print('Downloading STL-10 dataset for photo domain...')
stl10_train = STL10(root='/kaggle/working/stl10_data', split='train', download=True)

# Also try unlabeled split for more variety
stl10_extra = []
try:
    stl10_unlabeled = STL10(root='/kaggle/working/stl10_data', split='unlabeled', download=True)
    stl10_extra = [stl10_unlabeled[i][0] for i in range(min(2000, len(stl10_unlabeled)))]
    print(f'  STL-10 unlabeled: {len(stl10_extra)} extra photos')
except Exception as e:
    print(f'  STL-10 unlabeled failed: {e}')

photo_count = 0
max_photos = CONFIG['max_photos']

for i in range(min(len(stl10_train), max_photos)):
    img, _ = stl10_train[i]
    save_path = f'{stl10_dir}/{photo_count:05d}.png'
    img.save(save_path)
    domain_B_paths.append(save_path)
    photo_count += 1

for img in stl10_extra:
    if photo_count >= max_photos:
        break
    save_path = f'{stl10_dir}/{photo_count:05d}.png'
    img.save(save_path)
    domain_B_paths.append(save_path)
    photo_count += 1

print(f'\n>>> Total Domain B (Photos): {len(domain_B_paths)}')
print(f'\n{"="*50}')
print(f'Domain A (Sketches): {len(domain_A_paths)}')
print(f'Domain B (Photos):   {len(domain_B_paths)}')
print(f'{"="*50}')

In [ ]:
# ========================
# Preview samples from each domain
# ========================
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for i in range(5):
    img = Image.open(domain_A_paths[i]).convert('RGB')
    axes[0][i].imshow(img)
    axes[0][i].set_title(f'Sketch {i+1}', fontsize=10)
    axes[0][i].axis('off')

for i in range(5):
    img = Image.open(domain_B_paths[i]).convert('RGB')
    axes[1][i].imshow(img)
    axes[1][i].set_title(f'Photo {i+1}', fontsize=10)
    axes[1][i].axis('off')

axes[0][0].set_ylabel('Domain A\n(Sketch)', fontsize=12, fontweight='bold')
axes[1][0].set_ylabel('Domain B\n(Photo)', fontsize=12, fontweight='bold')
plt.suptitle('Dataset Preview', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ========================
# Dataset & DataLoaders
# ========================

class UnpairedDomainDataset(Dataset):
    """Unpaired dataset for CycleGAN. Domains are independently shuffled."""
    def __init__(self, paths_A, paths_B, transform=None):
        self.paths_A = paths_A
        self.paths_B = paths_B
        self.transform = transform

    def __getitem__(self, index):
        img_A = Image.open(self.paths_A[index % len(self.paths_A)]).convert('RGB')
        img_B = Image.open(self.paths_B[random.randint(0, len(self.paths_B) - 1)]).convert('RGB')
        if self.transform:
            img_A = self.transform(img_A)
            img_B = self.transform(img_B)
        return {'A': img_A, 'B': img_B}

    def __len__(self):
        return max(len(self.paths_A), len(self.paths_B))


transform_train = transforms.Compose([
    transforms.Resize((int(CONFIG['img_size'] * 1.12), int(CONFIG['img_size'] * 1.12))),
    transforms.RandomCrop(CONFIG['img_size']),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

transform_eval = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

# Shuffle & split 90/10
random.shuffle(domain_A_paths)
random.shuffle(domain_B_paths)
split = 0.9
train_A = domain_A_paths[:int(len(domain_A_paths) * split)]
val_A   = domain_A_paths[int(len(domain_A_paths) * split):]
train_B = domain_B_paths[:int(len(domain_B_paths) * split)]
val_B   = domain_B_paths[int(len(domain_B_paths) * split):]

train_dataset = UnpairedDomainDataset(train_A, train_B, transform=transform_train)
val_dataset   = UnpairedDomainDataset(val_A, val_B, transform=transform_eval)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True, num_workers=CONFIG['num_workers'],
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=CONFIG['num_workers'],
                          pin_memory=True)

print(f'Train: {len(train_A)} sketches, {len(train_B)} photos | {len(train_loader)} batches')
print(f'Val:   {len(val_A)} sketches, {len(val_B)} photos | {len(val_loader)} batches')

---
## 3. Model Architecture

### Generator: ResNet-based
```
c7s1-64 → d128 → d256 → R256×6 → u128 → u64 → c7s1-3
```
- ReflectionPad + InstanceNorm + ReLU
- 6 residual blocks (memory-efficient for T4)

### Discriminator: PatchGAN
```
C64 → C128 → C256 → C512 → 1-channel output
```
- Classifies overlapping patches as real/fake

### Loss Functions:
- LSGAN (MSE) for adversarial loss
- L1 for cycle consistency
- L1 for identity

### Optimizer:
- Adam with lr=0.0002, betas=(0.5, 0.999)
- Linear LR decay in second half of training

### Memory Optimization:
- Mixed Precision Training (torch.amp)
- DataParallel for dual T4 GPUs

In [ ]:
# ========================
# ResNet Generator
# ========================

class ResNetBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim),
        )

    def forward(self, x):
        return x + self.block(x)


class Generator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, ngf=64, n_blocks=6):
        super().__init__()
        # Encoder
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_ch, ngf, 7), nn.InstanceNorm2d(ngf), nn.ReLU(True),
        ]
        in_f = ngf
        for _ in range(2):
            out_f = in_f * 2
            model += [nn.Conv2d(in_f, out_f, 3, stride=2, padding=1),
                      nn.InstanceNorm2d(out_f), nn.ReLU(True)]
            in_f = out_f
        # Transformer
        for _ in range(n_blocks):
            model += [ResNetBlock(in_f)]
        # Decoder
        for _ in range(2):
            out_f = in_f // 2
            model += [nn.ConvTranspose2d(in_f, out_f, 3, stride=2, padding=1, output_padding=1),
                      nn.InstanceNorm2d(out_f), nn.ReLU(True)]
            in_f = out_f
        model += [nn.ReflectionPad2d(3), nn.Conv2d(ngf, out_ch, 7), nn.Tanh()]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


# Test
g = Generator(n_blocks=CONFIG['n_resnet_blocks'])
x = torch.randn(1, 3, CONFIG['img_size'], CONFIG['img_size'])
print(f'Generator: {sum(p.numel() for p in g.parameters()):,} params | output {g(x).shape}')
del g, x

In [ ]:
# ========================
# PatchGAN Discriminator
# ========================

class Discriminator(nn.Module):
    def __init__(self, in_ch=3, ndf=64):
        super().__init__()
        def block(in_f, out_f, norm=True):
            layers = [nn.Conv2d(in_f, out_f, 4, stride=2, padding=1)]
            if norm:
                layers.append(nn.InstanceNorm2d(out_f))
            layers.append(nn.LeakyReLU(0.2, True))
            return layers

        self.model = nn.Sequential(
            *block(in_ch, ndf, norm=False),
            *block(ndf, ndf*2),
            *block(ndf*2, ndf*4),
            nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(ndf*4, ndf*8, 4, stride=1, padding=1),
            nn.InstanceNorm2d(ndf*8),
            nn.LeakyReLU(0.2, True),
            nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(ndf*8, 1, 4, stride=1, padding=1),
        )

    def forward(self, x):
        return self.model(x)


d = Discriminator()
x = torch.randn(1, 3, CONFIG['img_size'], CONFIG['img_size'])
print(f'Discriminator: {sum(p.numel() for p in d.parameters()):,} params | output {d(x).shape}')
del d, x

In [ ]:
# ========================
# Weight Init, Replay Buffer, LR Scheduler
# ========================

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if hasattr(m, 'bias') and m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif 'Norm' in classname:
        if hasattr(m, 'weight') and m.weight is not None:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
            nn.init.constant_(m.bias.data, 0.0)


class ReplayBuffer:
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, data):
        result = []
        for element in data:
            el = element.unsqueeze(0).detach().clone()
            if len(self.data) < self.max_size:
                self.data.append(el)
                result.append(el)
            else:
                if random.random() > 0.5:
                    i = random.randint(0, self.max_size - 1)
                    result.append(self.data[i].clone())
                    self.data[i] = el
                else:
                    result.append(el)
        return torch.cat(result, 0)


def make_lr_scheduler(optimizer, cfg):
    def rule(epoch):
        return 1.0 - max(0, epoch - cfg['decay_epoch']) / float(
            cfg['num_epochs'] - cfg['decay_epoch'] + 1)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, rule)


print('Utilities defined: weights_init, ReplayBuffer, make_lr_scheduler')

In [ ]:
# ========================
# Initialize Models, Optimizers, Schedulers
# ========================

G_AB = Generator(CONFIG['in_channels'], CONFIG['out_channels'],
                 CONFIG['ngf'], CONFIG['n_resnet_blocks']).to(device)
G_BA = Generator(CONFIG['in_channels'], CONFIG['out_channels'],
                 CONFIG['ngf'], CONFIG['n_resnet_blocks']).to(device)
D_A = Discriminator(CONFIG['in_channels'], CONFIG['ndf']).to(device)
D_B = Discriminator(CONFIG['in_channels'], CONFIG['ndf']).to(device)

G_AB.apply(weights_init)
G_BA.apply(weights_init)
D_A.apply(weights_init)
D_B.apply(weights_init)

# DataParallel for dual GPUs
if n_gpus > 1:
    G_AB = nn.DataParallel(G_AB)
    G_BA = nn.DataParallel(G_BA)
    D_A  = nn.DataParallel(D_A)
    D_B  = nn.DataParallel(D_B)
    print(f'DataParallel enabled on {n_gpus} GPUs')

# Losses
criterion_GAN   = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_id    = nn.L1Loss()

# Optimizers
opt_G   = torch.optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()),
                           lr=CONFIG['lr'], betas=CONFIG['betas'])
opt_D_A = torch.optim.Adam(D_A.parameters(), lr=CONFIG['lr'], betas=CONFIG['betas'])
opt_D_B = torch.optim.Adam(D_B.parameters(), lr=CONFIG['lr'], betas=CONFIG['betas'])

# LR schedulers
sched_G   = make_lr_scheduler(opt_G, CONFIG)
sched_D_A = make_lr_scheduler(opt_D_A, CONFIG)
sched_D_B = make_lr_scheduler(opt_D_B, CONFIG)

# Replay buffers
buf_A = ReplayBuffer(CONFIG['buffer_size'])
buf_B = ReplayBuffer(CONFIG['buffer_size'])

# AMP scalers (modern API)
scaler_G = torch.amp.GradScaler('cuda')
scaler_D = torch.amp.GradScaler('cuda')

total_p = sum(sum(p.numel() for p in m.parameters()) for m in [G_AB, G_BA, D_A, D_B])
print(f'Total params: {total_p:,} ({total_p/1e6:.1f}M)')
if torch.cuda.is_available():
    print(f'VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

---
## 4. Training Loop

### Forward Pass:
1. Sketch → Photo (G_AB), Photo → Sketch (G_BA)
2. Cycle: Sketch → Photo → Sketch, Photo → Sketch → Photo
3. Identity: G_BA(sketch) ≈ sketch, G_AB(photo) ≈ photo

### Training Strategy:
- LSGAN (MSE loss) for stable training
- Mixed Precision with torch.amp
- Image replay buffer for discriminator stability
- Checkpoints every 10 epochs

In [ ]:
# ========================
# Training Helper: Save Samples
# ========================

def denorm(t):
    return (t * 0.5 + 0.5).clamp(0, 1)

def save_samples(epoch, rA, rB):
    G_AB.eval(); G_BA.eval()
    with torch.no_grad():
        fB = G_AB(rA[:4])
        fA = G_BA(rB[:4])
        recA = G_BA(fB[:4])
        recB = G_AB(fA[:4])
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    titles = ['Real Sketch', 'Fake Photo', 'Recovered Sketch', 'Real Photo']
    imgs = [rA[:4], fB[:4], recA[:4], rB[:4]]
    for c in range(4):
        for r in range(min(4, imgs[c].size(0))):
            im = denorm(imgs[c][r]).cpu().permute(1,2,0).numpy()
            axes[r][c].imshow(im); axes[r][c].axis('off')
            if r == 0: axes[r][c].set_title(titles[c], fontsize=12)
    plt.suptitle(f'Epoch {epoch+1}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{SAMPLE_DIR}/epoch_{epoch+1:04d}.png', dpi=100, bbox_inches='tight')
    plt.show(); plt.close()
    G_AB.train(); G_BA.train()

In [ ]:
# ========================
# Training Loop
# ========================

print('=' * 70)
print('Starting CycleGAN Training')
print(f'Epochs: {CONFIG["num_epochs"]} | Batch: {CONFIG["batch_size"]} | '
      f'Image: {CONFIG["img_size"]}x{CONFIG["img_size"]} | '
      f'GPUs: {n_gpus} | Batches/epoch: {len(train_loader)}')
print(f'LR: {CONFIG["lr"]} | Decay at: {CONFIG["decay_epoch"]} | '
      f'λ_cycle: {CONFIG["lambda_cycle"]} | λ_id: {CONFIG["lambda_identity"]}')
print('=' * 70)

history = {k: [] for k in ['G_loss','D_loss','cycle_loss','id_loss','adv_loss',
                            'D_A_loss','D_B_loss']}

for epoch in range(CONFIG['num_epochs']):
    G_AB.train(); G_BA.train(); D_A.train(); D_B.train()
    ep = {k: 0.0 for k in history}

    for i, batch in enumerate(train_loader):
        rA = batch['A'].to(device)
        rB = batch['B'].to(device)

        # --- Train Generators ---
        opt_G.zero_grad()
        with torch.amp.autocast('cuda'):
            # Identity
            id_A = G_BA(rA)
            id_B = G_AB(rB)
            l_id = (criterion_id(id_A, rA) + criterion_id(id_B, rB)) / 2
            # GAN
            fB = G_AB(rA)
            fA = G_BA(rB)
            l_gan_AB = criterion_GAN(D_B(fB), torch.ones_like(D_B(fB)))
            l_gan_BA = criterion_GAN(D_A(fA), torch.ones_like(D_A(fA)))
            l_gan = (l_gan_AB + l_gan_BA) / 2
            # Cycle
            l_cyc = (criterion_cycle(G_BA(fB), rA) + criterion_cycle(G_AB(fA), rB)) / 2
            # Total
            l_G = l_gan + CONFIG['lambda_cycle'] * l_cyc + CONFIG['lambda_identity'] * l_id

        scaler_G.scale(l_G).backward()
        scaler_G.step(opt_G)
        scaler_G.update()

        # --- Train D_A ---
        opt_D_A.zero_grad()
        with torch.amp.autocast('cuda'):
            fA_ = buf_A.push_and_pop(fA.detach())
            l_D_A = (criterion_GAN(D_A(rA), torch.ones_like(D_A(rA))) +
                     criterion_GAN(D_A(fA_), torch.zeros_like(D_A(fA_)))) / 2
        scaler_D.scale(l_D_A).backward()
        scaler_D.step(opt_D_A)
        scaler_D.update()

        # --- Train D_B ---
        opt_D_B.zero_grad()
        with torch.amp.autocast('cuda'):
            fB_ = buf_B.push_and_pop(fB.detach())
            l_D_B = (criterion_GAN(D_B(rB), torch.ones_like(D_B(rB))) +
                     criterion_GAN(D_B(fB_), torch.zeros_like(D_B(fB_)))) / 2
        scaler_D.scale(l_D_B).backward()
        scaler_D.step(opt_D_B)
        scaler_D.update()

        ep['G_loss'] += l_G.item()
        ep['D_loss'] += ((l_D_A + l_D_B) / 2).item()
        ep['cycle_loss'] += l_cyc.item()
        ep['id_loss'] += l_id.item()
        ep['adv_loss'] += l_gan.item()
        ep['D_A_loss'] += l_D_A.item()
        ep['D_B_loss'] += l_D_B.item()

    nb = len(train_loader)
    for k in ep:
        ep[k] /= nb
        history[k].append(ep[k])

    sched_G.step(); sched_D_A.step(); sched_D_B.step()

    if (epoch+1) % 5 == 0 or epoch == 0:
        lr = opt_G.param_groups[0]['lr']
        print(f'Ep [{epoch+1:3d}/{CONFIG["num_epochs"]}] '
              f'G:{ep["G_loss"]:.4f} D:{ep["D_loss"]:.4f} '
              f'Cyc:{ep["cycle_loss"]:.4f} Id:{ep["id_loss"]:.4f} LR:{lr:.6f}')

    if (epoch+1) % CONFIG['sample_interval'] == 0 or epoch == 0:
        save_samples(epoch, rA, rB)

    if (epoch+1) % CONFIG['checkpoint_interval'] == 0:
        # Unwrap DataParallel
        g_ab_sd = G_AB.module.state_dict() if hasattr(G_AB, 'module') else G_AB.state_dict()
        g_ba_sd = G_BA.module.state_dict() if hasattr(G_BA, 'module') else G_BA.state_dict()
        torch.save(g_ab_sd, f'{CHECKPOINT_DIR}/G_AB_epoch_{epoch+1}.pth')
        torch.save(g_ba_sd, f'{CHECKPOINT_DIR}/G_BA_epoch_{epoch+1}.pth')
        print(f'  -> Checkpoint saved at epoch {epoch+1}')

    if epoch == 0 and torch.cuda.is_available():
        print(f'  -> Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

# ========================
# Save Final Models (separate files, DataParallel unwrapped)
# ========================
g_ab_sd = G_AB.module.state_dict() if hasattr(G_AB, 'module') else G_AB.state_dict()
g_ba_sd = G_BA.module.state_dict() if hasattr(G_BA, 'module') else G_BA.state_dict()
torch.save(g_ab_sd, f'{CHECKPOINT_DIR}/cyclegan_G_AB_final.pth')
torch.save(g_ba_sd, f'{CHECKPOINT_DIR}/cyclegan_G_BA_final.pth')

print('\n' + '='*70)
print('Training Complete!')
print(f'Final models saved to {CHECKPOINT_DIR}/')
print(f'  cyclegan_G_AB_final.pth  (Sketch -> Photo)')
print(f'  cyclegan_G_BA_final.pth  (Photo -> Sketch)')
print('='*70)

---
## 5. Training Logs

Plot training losses across epochs.

In [ ]:
# ========================
# Plot Training Losses
# ========================
er = range(1, len(history['G_loss']) + 1)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0][0].plot(er, history['G_loss'], 'b-', lw=2, label='G Total')
axes[0][0].plot(er, history['adv_loss'], 'orange', alpha=.7, label='Adversarial')
axes[0][0].set_title('Generator Loss'); axes[0][0].legend(); axes[0][0].grid(alpha=.3)

axes[0][1].plot(er, history['D_loss'], 'r-', lw=2, label='D Total')
axes[0][1].plot(er, history['D_A_loss'], 'g-', alpha=.7, label='D_A')
axes[0][1].plot(er, history['D_B_loss'], 'purple', alpha=.7, label='D_B')
axes[0][1].set_title('Discriminator Loss'); axes[0][1].legend(); axes[0][1].grid(alpha=.3)

axes[1][0].plot(er, history['cycle_loss'], 'darkgreen', lw=2)
axes[1][0].set_title('Cycle Consistency Loss'); axes[1][0].grid(alpha=.3)

axes[1][1].plot(er, history['id_loss'], 'darkorange', lw=2)
axes[1][1].set_title('Identity Loss'); axes[1][1].grid(alpha=.3)

for ax in axes.flat:
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
plt.suptitle('CycleGAN Training Logs', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_losses.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Final — G: {history["G_loss"][-1]:.4f}  D: {history["D_loss"][-1]:.4f}  '
      f'Cycle: {history["cycle_loss"][-1]:.4f}  Identity: {history["id_loss"][-1]:.4f}')

---
## 6. Visualization Module

Display at least 5 qualitative examples showing:
- Input Sketch / Photo
- Generated Output (translated image)
- Reconstructed Image (cycle reconstruction)

In [ ]:
# ========================
# Qualitative Visualization
# ========================

def visualize(G_AB, G_BA, loader, n=5):
    G_AB.eval(); G_BA.eval()
    sks, phs = [], []
    for b in loader:
        sks.append(b['A']); phs.append(b['B'])
        if sum(s.size(0) for s in sks) >= n:
            break
    rS = torch.cat(sks)[:n].to(device)
    rP = torch.cat(phs)[:n].to(device)

    with torch.no_grad():
        fP = G_AB(rS); recS = G_BA(fP)
        fS = G_BA(rP); recP = G_AB(fS)

    # Sketch -> Photo -> Sketch
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    titles = ['Input Sketch', 'Generated Photo (G_AB)', 'Reconstructed Sketch']
    for i in range(n):
        for j, t in enumerate([rS[i], fP[i], recS[i]]):
            axes[i][j].imshow(denorm(t).cpu().permute(1,2,0).numpy())
            axes[i][j].axis('off')
            if i == 0: axes[i][j].set_title(titles[j], fontsize=12, fontweight='bold')
    plt.suptitle('Sketch → Photo Direction', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/viz_sketch_to_photo.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Photo -> Sketch -> Photo
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    titles = ['Input Photo', 'Generated Sketch (G_BA)', 'Reconstructed Photo']
    for i in range(n):
        for j, t in enumerate([rP[i], fS[i], recP[i]]):
            axes[i][j].imshow(denorm(t).cpu().permute(1,2,0).numpy())
            axes[i][j].axis('off')
            if i == 0: axes[i][j].set_title(titles[j], fontsize=12, fontweight='bold')
    plt.suptitle('Photo → Sketch Direction', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/viz_photo_to_sketch.png', dpi=150, bbox_inches='tight')
    plt.show()
    G_AB.train(); G_BA.train()

visualize(G_AB, G_BA, val_loader, n=5)

---
## 7. Quantitative Evaluation

Compute cycle reconstruction quality using:
- **SSIM (Structural Similarity Index)**
- **PSNR (Peak Signal-to-Noise Ratio)**

In [ ]:
# ========================
# SSIM & PSNR (Cycle Reconstruction)
# ========================

G_AB.eval(); G_BA.eval()
ssim_A, psnr_A = [], []
ssim_B, psnr_B = [], []

with torch.no_grad():
    for batch in val_loader:
        rA = batch['A'].to(device)
        rB = batch['B'].to(device)
        recA = G_BA(G_AB(rA))
        recB = G_AB(G_BA(rB))
        for i in range(rA.size(0)):
            oA = denorm(rA[i]).cpu().permute(1,2,0).numpy()
            rA_ = denorm(recA[i]).cpu().permute(1,2,0).numpy()
            oB = denorm(rB[i]).cpu().permute(1,2,0).numpy()
            rB_ = denorm(recB[i]).cpu().permute(1,2,0).numpy()
            ssim_A.append(ssim(oA, rA_, channel_axis=2, data_range=1.0))
            psnr_A.append(psnr(oA, rA_, data_range=1.0))
            ssim_B.append(ssim(oB, rB_, channel_axis=2, data_range=1.0))
            psnr_B.append(psnr(oB, rB_, data_range=1.0))
        if len(ssim_A) >= 50:
            break

print('='*60)
print('QUANTITATIVE EVALUATION — Cycle Reconstruction Quality')
print('='*60)
print(f'Sketch→Photo→Sketch:  SSIM={np.mean(ssim_A):.4f}  PSNR={np.mean(psnr_A):.2f} dB')
print(f'Photo→Sketch→Photo:   SSIM={np.mean(ssim_B):.4f}  PSNR={np.mean(psnr_B):.2f} dB')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(['Sketch Cycle', 'Photo Cycle'],
            [np.mean(ssim_A), np.mean(ssim_B)],
            yerr=[np.std(ssim_A), np.std(ssim_B)],
            color=['steelblue', 'darkorange'], capsize=10)
axes[0].set_title('SSIM', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 1); axes[0].grid(axis='y', alpha=.3)

axes[1].bar(['Sketch Cycle', 'Photo Cycle'],
            [np.mean(psnr_A), np.mean(psnr_B)],
            yerr=[np.std(psnr_A), np.std(psnr_B)],
            color=['steelblue', 'darkorange'], capsize=10)
axes[1].set_title('PSNR (dB)', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=.3)

plt.suptitle('Cycle Reconstruction Metrics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/evaluation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Gradio App Deployment

Interactive web app for real-time Sketch ↔ Photo translation.

In [ ]:
# ========================
# Gradio App
# ========================
import gradio as gr

G_AB.eval(); G_BA.eval()

def translate(input_image, direction):
    if input_image is None:
        return None
    img = Image.fromarray(input_image).convert('RGB')
    t = transform_eval(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = G_AB(t) if direction == 'Sketch to Photo' else G_BA(t)
    out = denorm(out[0]).cpu().permute(1,2,0).numpy()
    return (out * 255).astype(np.uint8)

demo = gr.Interface(
    fn=translate,
    inputs=[
        gr.Image(type='numpy', label='Input Image'),
        gr.Radio(['Sketch to Photo', 'Photo to Sketch'],
                 value='Sketch to Photo', label='Direction'),
    ],
    outputs=gr.Image(type='numpy', label='Output'),
    title='CycleGAN: Sketch ↔ Photo',
    description=(
        'Upload a sketch or photo and translate between domains.\n'
        'Model: ResNet Generator (6 blocks) + PatchGAN Discriminator\n'
        'Trained on: TU-Berlin + Sketchy + QuickDraw (sketches) and STL-10 (photos)'
    ),
    allow_flagging='never',
)

demo.launch(share=True, debug=False)
print('Gradio app launched!')